# Etapa 3 — Smoke Multicaso y Multi-Semilla (`smoke_all_cases_campaign`)

Replica `tools/smoke_all_cases_campaign.py` ejecutando **únicamente los modelos promovidos** desde `promoted_models.json`:
- Recorre el catálogo curado (`benchmarks` + `thesis_curated`, ajustable en `RUN_MODE`).
- Evalúa variabilidad entre semillas (`SEEDS = (0, 1, 2)` en `full_campaign`; `(0,)` en `demo_fast`) en `float32` con `adam_steps = 100`, `lbfgs_steps = 10`, caché **FEM WSL2** y reanudación opcional (`resume_from`).

In [1]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "demo_fast"

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(8, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='demo_fast' | USE_PARALLEL=True (workers=8, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
promo_file = CAMP_WS / "promoted_models.json"
if promo_file.is_file():
    PROMOTED = json.loads(promo_file.read_text(encoding="utf-8-sig"))["promoted_models"]
else:
    PROMOTED = ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"]

SEEDS = (0,) if RUN_MODE == "demo_fast" else (0, 1, 2)
CASE_IDS_S3 = (
    ["mms_constant", "mms_layered_y", "cable_disk", "xlpe_discrete_layers"]
    if RUN_MODE == "demo_fast"
    else [e["id"] for e in catalog.list_entries()]
)
print("Modelos promovidos:", PROMOTED, "| Casos:", len(CASE_IDS_S3), "| Semillas:", SEEDS)

Modelos promovidos: ['M1', 'M1_EXACT', 'CUSPR', 'M0R', 'M2'] | Casos: 4 | Semillas: (0,)


In [3]:
stage3_dir = CAMP_WS / "stage3_multicase_smoke"
stage3_dir.mkdir(parents=True, exist_ok=True)
summary_path = stage3_dir / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8-sig")) if summary_path.is_file() else {}
tasks_s3 = []
task_meta_s3 = {}

for case_id in CASE_IDS_S3:
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in PROMOTED if RUN_MODE == "full_campaign" else PROMOTED[:3]:
        for seed in SEEDS:
            key = f"{case_id}:{arch}:seed{seed}"
            if summary.get(key, {}).get("status") == "completed":
                continue
            out_dir = stage3_dir / case_id / arch / f"seed_{seed}"
            task_meta_s3[key] = (case_id, arch, seed, ref_lbl)
            tasks_s3.append(
                RunTask(
                    name=key,
                    problem_source=prob.doc,
                    architecture=arch,
                    output_dir=out_dir,
                    config_kwargs={
                        "adam_steps": 60 if RUN_MODE == "demo_fast" else 100,
                        "lbfgs_steps": 5 if RUN_MODE == "demo_fast" else 10,
                        "lbfgs_iterations": 1,
                        "lr": 0.003,
                        "width": 32,
                        "depth": 3,
                        "dtype": "float32",
                        "device": DEVICE,
                        "init_seed": seed,
                        "sampling_seed": 11 + seed,
                        "weights": {"pde": 1.0, "energy": 100.0},
                        "points": (
                            {"n_interior": 512, "n_layer": 256, "n_face": 128, "n_boundary": 128}
                            if RUN_MODE == "demo_fast"
                            else {"n_interior": 1024, "n_layer": 512, "n_face": 256, "n_boundary": 256}
                        ),
                        "sampling": "mixed",
                        "focused": True,
                        "focus_criterion": "residual",
                        "focus_surfaces": False,
                        "resample_every": 100,
                    },
                    reference=ref_vals,
                    reference_label=ref_lbl,
                    overwrite=True,
                )
            )

def _on_stage3_done(res: dict) -> None:
    key = res["name"]
    case_id, arch, seed, ref_lbl = task_meta_s3[key]
    summary[key] = {
        "case": case_id,
        "architecture": arch,
        "seed": seed,
        "status": res.get("status"),
        "reference_label": ref_lbl,
        "smoke_only": True,
        **extract_metrics(res),
    }
    if res.get("error"):
        summary[key]["error"] = res.get("error")
    write_json_atomic(summary_path, summary)
    print(f"[Etapa 3] {key:36s} | dev={res.get('device')} | bal={summary[key]['balance_pct']}% | NRMSE={summary[key]['nrmse_pct']}%")

runner_s3 = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
runner_s3.run(tasks_s3, on_complete=_on_stage3_done)

[Etapa 3] mms_constant:M1:seed0                | dev=cpu | bal=34.70742408451937% | NRMSE=23.09346367644126%
[Etapa 3] mms_constant:CUSPR:seed0             | dev=cpu | bal=11.612606464162496% | NRMSE=11.597178605232587%
[Etapa 3] mms_layered_y:M1:seed0               | dev=cpu | bal=28.976546784911548% | NRMSE=24.018095835132595%
[Etapa 3] cable_disk:M1:seed0                  | dev=cpu | bal=0.9416655917870971% | NRMSE=27.74382508900264%
[Etapa 3] mms_constant:M1_EXACT:seed0          | dev=cpu | bal=0.8652810580290841% | NRMSE=1.7259649183826782%
[Etapa 3] mms_layered_y:CUSPR:seed0            | dev=cpu | bal=33.65414726280124% | NRMSE=14.052700688717394%
[Etapa 3] mms_layered_y:M1_EXACT:seed0         | dev=cpu | bal=2.425517010598485% | NRMSE=0.9221774711892187%
[Etapa 3] cable_disk:CUSPR:seed0               | dev=cpu | bal=17.14267534105129% | NRMSE=27.88788550976871%
[Etapa 3] cable_disk:M1_EXACT:seed0            | dev=cpu | bal=39.60647951548427% | NRMSE=24.94429680583031%
[Etapa 3] 

[{'name': 'mms_constant:M1:seed0',
  'status': 'completed',
  'output_dir': 'C:\\usr\\ths_mia_fiis\\tesis_MIA_2026-2_pinn\\notebooks_campaign\\campaign_workspace\\stage3_multicase_smoke\\mms_constant\\M1\\seed_0',
  'architecture': 'M1',
  'case': 'mms_constant',
  'device': 'cpu',
  'parameters': 2242,
  'best_loss': 35.10334014892578,
  'accuracy_stop': None,
  'elapsed_s': 6.915740099968389,
  'evaluation': {'pde_rms': {'domain': 0.6428244709968567},
   'constitutive_rms': {},
   'interfaces': [],
   'boundary': [{'side': 'bottom',
     'region': 'domain',
     'type': 'dirichlet',
     'rms': 6.089927673339844,
     'unit': 'K'},
    {'side': 'top',
     'region': 'domain',
     'type': 'dirichlet',
     'rms': 6.147999286651611,
     'unit': 'K'},
    {'side': 'left',
     'region': 'domain',
     'type': 'dirichlet',
     'rms': 8.335901260375977,
     'unit': 'K'},
    {'side': 'right',
     'region': 'domain',
     'type': 'dirichlet',
     'rms': 7.429039478302002,
     'unit'

In [4]:
import pandas as pd

df_s3 = pd.DataFrame(summary).T
df_s3 = df_s3[df_s3["status"] == "completed"].copy()
num_cols = ["error_Tmax_abs_K", "nrmse_pct", "balance_pct"]
df_s3[num_cols] = df_s3[num_cols].astype(float)

def cumple(row, crit):
    return all(row[k] <= crit[k] for k in crit)

df_s3["cumple_estricto"] = df_s3.apply(cumple, axis=1, crit=STRICT_CRITERIA)
df_s3["cumple_relajado"] = df_s3.apply(cumple, axis=1, crit=RELAXED_CRITERIA)

ranking = (
    df_s3.groupby("architecture")
    .agg(
        casos=("case", "count"),
        nrmse_medio=("nrmse_pct", "mean"),
        err_Tmax_medio_K=("error_Tmax_abs_K", "mean"),
        balance_medio_pct=("balance_pct", "mean"),
        cumple_relajado=("cumple_relajado", "sum"),
    )
    .sort_values("nrmse_medio")
)
ranking.insert(0, "rank", range(1, len(ranking) + 1))
print(ranking.round(3))

mejor_por_caso = df_s3.loc[
    df_s3.groupby("case")["nrmse_pct"].idxmin(),
    ["case", "architecture", "nrmse_pct"],
]
print("\nMejor modelo por caso (NRMSE):")
print(mejor_por_caso.to_string(index=False))

              rank  casos  nrmse_medio  err_Tmax_medio_K  balance_medio_pct  \
architecture                                                                  
M1_EXACT         1      4        8.816            29.571            169.941   
M1               2      4       20.342            36.943            276.202   
CUSPR            3      4       20.547            33.981           1182.497   

              cumple_relajado  
architecture                   
M1_EXACT                    0  
M1                          0  
CUSPR                       0  

Mejor modelo por caso (NRMSE):
                case architecture  nrmse_pct
          cable_disk     M1_EXACT  24.944297
        mms_constant     M1_EXACT   1.725965
       mms_layered_y     M1_EXACT   0.922177
xlpe_discrete_layers           M1   6.513250
